# Day 5: vLLM Multi-LoRA Serving & FastAPI Gateway
### **Routed Multi-Adapter LLM Serving System**

This notebook launches the production serving architecture on an NVIDIA T4 GPU:
1. **vLLM Engine (Port 8000):** Hosts frozen `Qwen/Qwen2.5-1.5B-Instruct` base model + 3 dynamically activated LoRA adapters (`sql-adapter`, `json-adapter`, `code-adapter`).
2. **FastAPI Gateway (Port 8080):** Intelligent semantic routing layer classifying incoming queries via `fastembed` in ~2.5ms.
3. **pyngrok Tunnel:** Exposes the Gateway endpoint publicly so you can send queries and benchmark from your local machine.

## 1. Verify GPU Hardware Acceleration

In [ ]:
!nvidia-smi

## 2. Install Serving Dependencies

In [ ]:
!pip install -q "vllm>=0.6.0" "fastapi>=0.110.0" "uvicorn>=0.28.0" "pyngrok>=7.1.0" "fastembed>=0.2.0" requests

## 3. Verify Adapter Checkpoints
Ensure `adapters/sql_lora`, `adapters/json_lora`, and `adapters/code_lora` are present.

In [ ]:
import os

# If you have adapters.zip, unpack it:
if os.path.exists('adapters.zip') and not os.path.exists('adapters/sql_lora'):
    !unzip -q adapters.zip

required_adapters = ['adapters/sql_lora', 'adapters/json_lora', 'adapters/code_lora']
for a in required_adapters:
    assert os.path.exists(a), f'Missing adapter checkpoint: {a}'
print('All 3 LoRA adapter checkpoints are verified and ready.')

## 4. Launch vLLM Multi-LoRA Engine (Port 8000)
Configured for NVIDIA T4: `--dtype float16`, `--enforce-eager`, `--max-model-len 2048`, and `--gpu-memory-utilization 0.80`.

In [ ]:
%%bash --bg
python3 -m vllm.entrypoints.openai.api_server \
    --model Qwen/Qwen2.5-1.5B-Instruct \
    --dtype float16 \
    --enforce-eager \
    --max-model-len 2048 \
    --enable-lora \
    --lora-modules \
        sql-adapter=adapters/sql_lora \
        json-adapter=adapters/json_lora \
        code-adapter=adapters/code_lora \
    --max-loras 3 \
    --max-lora-rank 16 \
    --gpu-memory-utilization 0.80 \
    --port 8000 > vllm.log 2>&1

## 5. Wait for vLLM Engine Initialization
Polls the vLLM `/health` endpoint until the server completes model loading and KV-cache allocation.

In [ ]:
import time, requests

print('Waiting for vLLM server to become healthy (typically 30-45s)...')
vllm_ready = False
for attempt in range(60):
    try:
        resp = requests.get('http://localhost:8000/health', timeout=1.0)
        if resp.status_code == 200:
            vllm_ready = True
            print(f'vLLM Engine is ONLINE and HEALTHY (ready in ~{attempt * 2}s)!')
            break
    except requests.exceptions.RequestException:
        pass
    time.sleep(2.0)

if not vllm_ready:
    print('ERROR: vLLM failed to start. Printing last 30 lines of vllm.log:')
    !tail -n 30 vllm.log

## 6. Launch FastAPI Semantic Gateway & Public Tunnel (Port 8080)
Starts the routing gateway and optionally creates a public HTTPS tunnel with `pyngrok`.

In [ ]:
# Enter your free ngrok auth token from https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = ''  # Optional: leave empty if testing locally inside Colab

import threading
from src.gateway import start_gateway

# Start FastAPI gateway in background thread
gateway_thread = threading.Thread(
    target=start_gateway,
    kwargs={'port': 8080, 'host': '0.0.0.0', 'use_ngrok': bool(NGROK_TOKEN), 'ngrok_token': NGROK_TOKEN or None},
    daemon=True
)
gateway_thread.start()
time.sleep(3.0)
print('Gateway started on port 8080.')

## 7. Run Automated Smoke Test Client
Sends 4 test queries to verify live semantic routing and multi-LoRA inference.

In [ ]:
!python scripts/smoke_test_gateway.py --url http://localhost:8080